# DSPy Mission Control

PyHou, May 19 2026. Ahliana Byrd.

This notebook is a companion to the live talk. Cells run top to bottom. Cell 6 is the placeholder for the partner activity, planned to go with a real demo of your choice.

***Before running anything, create an `.env` and add your `ANTHROPIC_API_KEY`***.

## Cell 1: Setup

Configure DSPy to talk to Claude Haiku 4.5.

In [1]:
import os
import logging
from dotenv import load_dotenv

# Quiet noisy LiteLLM warnings about AWS Bedrock/SageMaker pre-loading.
# DSPy uses LiteLLM internally and LiteLLM tries to pre-load handlers for
# every provider. Without botocore installed those warnings fire on import,
# but we are not using AWS here, so silence them before importing dspy.
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

import dspy

load_dotenv(override=True)
lm = dspy.LM("anthropic/claude-haiku-4-5-20251001", max_tokens=800)
dspy.configure(lm=lm)
print("Mission Control online.")

Mission Control online.


## Cell 2: Your First Signature

A signature is a specification. Inputs go before the arrow, outputs go after. That is the whole concept. No prompt was written.

In [2]:
classify = dspy.Predict("text -> sentiment: str")
result = classify(text="The launch was successful and the crew is in great spirits.")
print(result.sentiment)

positive


## Cell 3: The Reveal

This is what DSPy actually sent to the LM. The user never wrote a prompt. The library wrote it from the user's signature.

In [3]:
dspy.inspect_history(n=1)





[2026-05-18T23:19:36.086738]

System message:

Your input fields are:
1. `text` (str):
Your output fields are:
1. `sentiment` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `text`, produce the fields `sentiment`.


User message:

[[ ## text ## ]]
The launch was successful and the crew is in great spirits.

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
positive

[[ ## completed ## ]]







## Cell 4: Class-Form Signatures

Same idea, more structure. Typed fields, descriptions, a docstring.

In [4]:
class MissionReport(dspy.Signature):
    """Classify a mission status update from a flight controller."""
    transmission: str = dspy.InputField()
    severity: str = dspy.OutputField(desc="One of: nominal, advisory, urgent")
    summary: str = dspy.OutputField(desc="One sentence summary")

report = dspy.Predict(MissionReport)
out = report(transmission="Houston, we have a problem. Main bus B undervolt.")
print(f"Severity: {out.severity}")
print(f"Summary: {out.summary}")

Severity: urgent
Summary: Main bus B is experiencing undervoltage, indicating a critical power system failure requiring immediate attention.


## Cell 5: Signatures Do Not Police Your Intent

Sometimes the model just goes for it.

In [5]:
vibes = dspy.Predict("vibes -> stock_pick: str, confidence: float")
print(vibes(vibes="kind of a mercury retrograde tuesday energy"))

Prediction(
    stock_pick='NVDA',
    confidence=0.23
)


## Cell 6: Partner Activity (Live demo)

The user will type a real signature here, the user's choice.

Example card: "input: meeting transcript, output: action_items as a list of strings"


In [6]:
# Type the user's signature, then run it
# Example placeholder (replace with the actual demo):

class AttendeeMission(dspy.Signature):
    """Extract action items from a meeting transcript."""
    transcript: str = dspy.InputField()
    action_items: str = dspy.OutputField(desc="Newline-separated list of action items")

demo = dspy.Predict(AttendeeMission)
result = demo(transcript="Sarah will draft the proposal by Friday. Mike will review the budget by Wednesday. Everyone needs to update their availability for next week.")
print(result.action_items)

Sarah will draft the proposal by Friday
Mike will review the budget by Wednesday
Everyone needs to update their availability for next week


## Cell 7: Modules

Signatures describe WHAT. Modules describe HOW. Same signature, different module, different behavior. The reasoning field appears automatically with ChainOfThought.

In [7]:
class ExplainCode(dspy.Signature):
    """Explain what this Python code does in plain English."""
    code: str = dspy.InputField()
    explanation: str = dspy.OutputField()

snippet = "def fib(n):\n    return n if n <= 1 else fib(n-1) + fib(n-2)"

print("Predict says:")
print(dspy.Predict(ExplainCode)(code=snippet).explanation)
print("\nChainOfThought says:")
result = dspy.ChainOfThought(ExplainCode)(code=snippet)
print(f"Reasoning: {result.reasoning}\n")
print(f"Explanation: {result.explanation}")

Predict says:
This code defines a function called `fib` that calculates the nth number in the Fibonacci sequence using recursion.

Here's how it works:
- It takes a single parameter `n` (a non-negative integer)
- If `n` is less than or equal to 1, it returns `n` directly (base case: fib(0)=0, fib(1)=1)
- Otherwise, it recursively calls itself with `n-1` and `n-2`, then adds those two results together

For example:
- `fib(0)` returns 0
- `fib(1)` returns 1
- `fib(5)` would return 5 by calculating: fib(4) + fib(3), which eventually breaks down into the base cases and sums to 5

The Fibonacci sequence is: 0, 1, 1, 2, 3, 5, 8, 13, 21...

**Note:** While this code is elegant and mathematically correct, it's inefficient for larger values of `n` because it recalculates the same values many times (exponential time complexity).

ChainOfThought says:
Reasoning: This is a recursive function that implements the Fibonacci sequence. The function uses a conditional expression (ternary operator) to ch

## Cell 7b: ReAct, the Tool-Calling Module

Predict answers in one shot. ChainOfThought reasons first, then answers. ReAct calls tools you give it, sees the results, and iterates until it has an answer.

Same signature pattern, different module, different capability.

In [8]:
import datetime

def days_since(date_str: str) -> str:
    """Return how many days have passed since the given ISO 8601 date (YYYY-MM-DD)."""
    past = datetime.date.fromisoformat(date_str)
    return f"{(datetime.date.today() - past).days} days"

class MissionTimer(dspy.Signature):
    """Answer questions involving dates and elapsed time, using the provided tool when needed."""
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

agent = dspy.ReAct(MissionTimer, tools=[days_since])
result = agent(question="Apollo 11 launched on 1969-07-16. How many days ago was that, and what does that tell us about the gap between Apollo and Artemis?")
print(result.answer)

Apollo 11 launched approximately 20,760 days (or roughly 56.8 years) ago. This substantial gap tells us that there is more than half a century between the Apollo era of lunar exploration and the current Artemis program. This extended interval highlights how long humanity paused its efforts to return to the Moon after the successful Apollo missions, and underscores the significance of the Artemis program as a historic resumption of human lunar exploration after such a lengthy hiatus.


## Cell 8: A Real Module, the Code Reviewer

A signature plus ChainOfThought equals a working code review module. Quality score, issues, fixed code, all in one call.

In [9]:
class CodeReviewSig(dspy.Signature):
    """Review Python code and return structured findings."""
    code: str = dspy.InputField()
    quality: int = dspy.OutputField(desc="Quality score from 1 to 10")
    issues: str = dspy.OutputField(desc="Newline-separated list of specific issues")
    improved: str = dspy.OutputField(desc="The code rewritten with the issues fixed")

reviewer = dspy.ChainOfThought(CodeReviewSig)

code_to_review = """
def calculate_total(items):
    total = 0
    for item in items:
        total = total + item.price
    return total
"""

result = reviewer(code=code_to_review)
print(f"Quality: {result.quality}/10")
print(f"Issues:\n{result.issues}")
print(f"\nImproved:\n{result.improved}")

Quality: 5/10
Issues:
No docstring or type hints provided
No input validation or error handling
Not Pythonic - uses manual accumulation instead of built-in sum()
No handling of edge cases (empty list, None items, missing price attribute)
Function doesn't validate that items have a price attribute before accessing it

Improved:
def calculate_total(items):
    """
    Calculate the total price of items.
    
    Args:
        items: An iterable of objects with a 'price' attribute
        
    Returns:
        float: The sum of all item prices
        
    Raises:
        TypeError: If items is not iterable or items lack a price attribute
        ValueError: If any price is not a number
    """
    if not items:
        return 0
    
    try:
        return sum(item.price for item in items)
    except AttributeError as e:
        raise AttributeError(f"Item missing 'price' attribute: {e}") from e
    except TypeError as e:
        raise TypeError(f"Price must be a number: {e}") from e


## Cell 9: Swap Models in One Line

Same module, bigger model. The signature does not change. The module does not change. The user edits one line.

In [10]:
sonnet = dspy.LM("anthropic/claude-sonnet-4-6", max_tokens=1000)
with dspy.context(lm=sonnet):
    sonnet_result = reviewer(code=code_to_review)

print("Haiku 4.5 said:")
print(result.improved)
print("\n---\n")
print("Sonnet 4.6 said:")
print(sonnet_result.improved)

Haiku 4.5 said:
def calculate_total(items):
    """
    Calculate the total price of items.
    
    Args:
        items: An iterable of objects with a 'price' attribute
        
    Returns:
        float: The sum of all item prices
        
    Raises:
        TypeError: If items is not iterable or items lack a price attribute
        ValueError: If any price is not a number
    """
    if not items:
        return 0
    
    try:
        return sum(item.price for item in items)
    except AttributeError as e:
        raise AttributeError(f"Item missing 'price' attribute: {e}") from e
    except TypeError as e:
        raise TypeError(f"Price must be a number: {e}") from e

---

Sonnet 4.6 said:
from typing import Protocol, Sequence


class Priceable(Protocol):
    price: float


def calculate_total(items: Sequence[Priceable]) -> float:
    """Calculate the total price of all items.

    Args:
        items: A sequence of objects with a `price` attribute.

    Returns:
        The su

## Cell 10: The Live Optimizer

You bring training examples and a metric. DSPy tunes the prompt for you. No more hand-tuning by trial and error.

This runs in under 90 seconds. Baseline first, then the optimizer compiles a tuned version using 12 training examples. After optimization, `inspect_history` shows the demonstrations DSPy added to the prompt automatically.

In [11]:
class ClassifyTransmission(dspy.Signature):
    """Classify a mission transmission by urgency."""
    transmission: str = dspy.InputField()
    urgency: str = dspy.OutputField(desc="One of: routine, attention, urgent")

train = [
    # Clear-cut cases (the original 6)
    dspy.Example(transmission="Routine status check, all systems nominal.", urgency="routine").with_inputs("transmission"),
    dspy.Example(transmission="Minor cabin pressure fluctuation detected.", urgency="attention").with_inputs("transmission"),
    dspy.Example(transmission="Main bus B undervolt, immediate response required.", urgency="urgent").with_inputs("transmission"),
    dspy.Example(transmission="Daily science experiment completed on schedule.", urgency="routine").with_inputs("transmission"),
    dspy.Example(transmission="Slight trajectory deviation, investigating cause.", urgency="attention").with_inputs("transmission"),
    dspy.Example(transmission="Cabin fire detected, crew evacuating to lifeboat.", urgency="urgent").with_inputs("transmission"),
    # Ambiguous cases without obvious urgency words (force the optimizer to learn nuance)
    dspy.Example(transmission="Solar array pointing error 0.3 degrees, automatic correction failed twice.", urgency="attention").with_inputs("transmission"),
    dspy.Example(transmission="Crew morale low after fourth straight day of docking delays.", urgency="attention").with_inputs("transmission"),
    dspy.Example(transmission="Coolant loop B isolation valve will not close, loop A still nominal.", urgency="urgent").with_inputs("transmission"),
    dspy.Example(transmission="Hydroponics shelf 4 humidity 92 percent, dehumidifier running but not catching up.", urgency="attention").with_inputs("transmission"),
    dspy.Example(transmission="Cabin air filter scheduled replacement complete, performance nominal.", urgency="routine").with_inputs("transmission"),
    dspy.Example(transmission="Robotic arm joint 3 actuator drawing 15 percent above baseline current.", urgency="attention").with_inputs("transmission"),
]

def match(ex, pred, trace=None):
    return ex.urgency.strip().lower() == pred.urgency.strip().lower()

test_case = "Houston, the optimizer is taking longer than expected."

# Baseline, no optimization
baseline = dspy.Predict(ClassifyTransmission)
print("Baseline result:", baseline(transmission=test_case).urgency)
print("\n--- Baseline prompt sent to the LM ---")
dspy.inspect_history(n=1)

Baseline result: attention

--- Baseline prompt sent to the LM ---




[2026-05-18T23:19:44.048524]

System message:

Your input fields are:
1. `transmission` (str):
Your output fields are:
1. `urgency` (str): One of: routine, attention, urgent
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## transmission ## ]]
{transmission}

[[ ## urgency ## ]]
{urgency}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify a mission transmission by urgency.


User message:

[[ ## transmission ## ]]
Houston, the optimizer is taking longer than expected.

Respond with the corresponding output fields, starting with the field `[[ ## urgency ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## urgency ## ]]
attention

[[ ## completed ## ]]







Now optimize with BootstrapFewShot. This will run in under 90 seconds.

In [12]:
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(metric=match, max_bootstrapped_demos=3)
optimized = optimizer.compile(student=dspy.Predict(ClassifyTransmission), trainset=train)

print("Optimized result:", optimized(transmission=test_case).urgency)
print("\n--- Optimized prompt sent to the LM ---")
dspy.inspect_history(n=1)

Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Optimized result: attention

--- Optimized prompt sent to the LM ---




[2026-05-18T23:19:44.087254]

System message:

Your input fields are:
1. `transmission` (str):
Your output fields are:
1. `urgency` (str): One of: routine, attention, urgent
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## transmission ## ]]
{transmission}

[[ ## urgency ## ]]
{urgency}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify a mission transmission by urgency.


User message:

[[ ## transmission ## ]]
Routine status check, all systems nominal.


Assistant message:

[[ ## urgency ## ]]
routine

[[ ## completed ## ]]


User message:

[[ ## transmission ## ]]
Minor cabin pressure fluctuation detected.


Assistant message:

[[ ## urgency ## ]]
attention

[[ ## completed ## ]]


User message:

[[ ## transmission ## ]]
Main bus 

## Cell 11: A More Sophisticated Optimizer (Pre-Run)

BootstrapFewShot is fast and works. There are heavier optimizers that search more carefully. MIPROv2 uses Bayesian optimization over instruction proposals and demonstration sets. It takes 30 to 45 minutes to compile but typically produces stronger results.

I ran MIPROv2 (or BootstrapFewShotWithRandomSearch as a fallback) previously and saved the optimized program to `miprov2_artifact.json`. Load and run it now.

In [13]:
# This artifact was generated offline. The cell below shows the production pattern:
# optimize once, save, load and deploy.
import os

if not os.path.exists("miprov2_artifact.json"):
    print("ERROR: miprov2_artifact.json not found.")
    print("Run the Sunday prep cell below to generate it before the talk.")
else:
    sophisticated = dspy.Predict(ClassifyTransmission)
    sophisticated.load("miprov2_artifact.json")

    test_cases = [
        "Houston, the optimizer is taking longer than expected.",
        "Crew reports unusual noise from the hydroponics module.",
        "All systems green, scheduled meal break commencing.",
    ]

    for t in test_cases:
        result = sophisticated(transmission=t)
        print(f"[{result.urgency:>9}]  {t}")

    print("\n--- The prompt the heavier optimizer produced ---")
    dspy.inspect_history(n=1)

[attention]  Houston, the optimizer is taking longer than expected.
[attention]  Crew reports unusual noise from the hydroponics module.
[  routine]  All systems green, scheduled meal break commencing.

--- The prompt the heavier optimizer produced ---




[2026-05-18T23:19:44.111627]

System message:

Your input fields are:
1. `transmission` (str):
Your output fields are:
1. `urgency` (str): One of: routine, attention, urgent
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## transmission ## ]]
{transmission}

[[ ## urgency ## ]]
{urgency}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify a mission transmission by urgency.


User message:

[[ ## transmission ## ]]
Minor cabin pressure fluctuation detected.


Assistant message:

[[ ## urgency ## ]]
attention

[[ ## completed ## ]]


User message:

[[ ## transmission ## ]]
Main bus B undervolt, immediate response required.


Assistant message:


## Previously Prepared Cell

This is NOT run during the presentation, it is prepared beforehand. It can take 30 to 45 minutes, or much less, for MIPROv2. If MIPROv2 errors, the cell falls back to BootstrapFewShotWithRandomSearch (5 to 10 minutes). Either way, you get `miprov2_artifact.json` saved to disk, which cell 11 loads during the live presentation.

In [ ]:
# Runtime: 30-45 min for MIPROv2, 5-10 min for the fallback.

import os

def match(ex, pred, trace=None):
    return ex.urgency.strip().lower() == pred.urgency.strip().lower()

# This block tries MIPROv2 first, falls back to BSRS if it errors.

try:
    from dspy.teleprompt import MIPROv2
    print("Compiling with MIPROv2 (this takes 30-45 minutes)...")
    miprov2 = MIPROv2(metric=match, auto="light", num_threads=4)
    artifact = miprov2.compile(
        student=dspy.Predict(ClassifyTransmission),
        trainset=train,
    )
    print("MIPROv2 succeeded.")
except Exception as e:
    print(f"MIPROv2 errored: {e}")
    print("Falling back to BootstrapFewShotWithRandomSearch...")
    from dspy.teleprompt import BootstrapFewShotWithRandomSearch
    bsrs = BootstrapFewShotWithRandomSearch(
        metric=match,
        max_bootstrapped_demos=4,
        num_candidate_programs=6,
    )
    artifact = bsrs.compile(
        student=dspy.Predict(ClassifyTransmission),
        trainset=train,
    )
    print("BSRS succeeded.")

artifact.save("miprov2_artifact.json")
print(f"\nSaved to miprov2_artifact.json ({os.path.getsize('miprov2_artifact.json')} bytes)")
print("Verify by running cell 11 above.")

## Mission complete

The repo: github.com/ahliana/dspy_mission_control

Continue the mission on the PyHou Discord. Find your crew on LinkedIn.

What could YOU teach in 5 minutes? Sign up for a lightning talk.

---
